## Импорты

In [1]:
!pip install natasha sentence-transformers pymorphy3

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.4/34.4 MB 35.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.5/55.5 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 77.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.2/8.2 MB 65.2 MB/s eta 0:00:00
  Created wheel for docopt: filename=docopt-0.6.2-py2.py3-none-any.whl size=13706 sha256=1f015e6b666cb75e20c50937cae8ffc9741550a8cd171373b3975db8d0ccdf25
  Stored in directory: /root/.cache/pip/wheels/1a/bf/a1/4cee4f7678c68c5875ca89eaccf460593539805c3906722228
Successfully built docopt


In [2]:
import re
import pickle
import random
import numpy as np
import pandas as pd
from tqdm import tqdm
from collections import defaultdict
from natasha import (
    Segmenter,
    NewsNERTagger,
    NewsEmbedding,
    NewsMorphTagger,
    NewsSyntaxParser,
    MorphVocab,
    Doc
)
import pymorphy3

In [3]:
from google.colab import files

In [4]:
random.seed(42)

## Подготовка данных

In [ ]:
with open('/content/Doctor_Piter.txt', 'r', encoding='utf-8') as f:
    content = f.read()
raw_docs = content.split('=====')
raw_docs = [doc for doc in raw_docs if not '\n\n' in doc]

In [ ]:
def parse_document(text):
    """
    Парсит один документ из формата:
    https://url
    Источник
    Дата
    Автор
    Заголовок
    Текст...
    """
    lines = text.strip().split('\n')
    lines = [l.strip() for l in lines if l.strip()]

    if len(lines) < 5:
        return None

    for line in lines:
        if line == '':
            return None

    doc = {
        'url': lines[0].replace('\n', '').replace('. ru', '.ru').strip() if lines[0].startswith('http') else '',
        'source': lines[1].replace('\n', '').strip() if len(lines) > 1 else '',
        'date': lines[2].replace('\n', '').strip() if len(lines) > 2 else '',
        'author': lines[3].replace('\n', '').strip() if len(lines) > 3 else '',
        'title': lines[4].replace('\n', '').strip() if len(lines) > 4 else '',
        'content': ' '.join(lines[5:]).replace('\n', '').strip() if len(lines) > 5 else '',
    }

    return doc

# Применяем ко всем документам
parsed_docs = []
for raw_doc in tqdm(raw_docs):
    doc = parse_document(raw_doc)
    if doc:
        parsed_docs.append(doc)

100%|██████████| 3098/3098 [00:00<00:00, 73498.54it/s]


In [ ]:
docs = random.sample(parsed_docs, 750)

In [ ]:
with open('documents.pkl', 'wb') as f:
    pickle.dump(docs, f)
files.download('documents.pkl')
print("Документы сохранены в documents.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Документы сохранены в documents.pkl


## Выделение сущностей

In [ ]:
segmenter = Segmenter()
emb = NewsEmbedding()
morph_tagger = NewsMorphTagger(emb)
syntax_parser = NewsSyntaxParser(emb)
ner_tagger = NewsNERTagger(emb)
morph_vocab = MorphVocab()
morph = pymorphy3.MorphAnalyzer()

In [ ]:
def extract_entities_from_text(text):
    """
    Извлекает именованные сущности с нормализацией
    """
    if not text or len(text) < 10:
        return []

    try:
        # Создаем документ
        doc = Doc(text)
        doc.segment(segmenter)
        doc.tag_morph(morph_tagger)
        doc.parse_syntax(syntax_parser)
        doc.tag_ner(ner_tagger)

        # Нормализуем сущности
        for span in doc.spans:
            span.normalize(morph_vocab)

        # Собираем результаты
        entities = []
        seen = set()  # для уникальности

        for span in doc.spans:
            if span.type in ['ORG', 'PER', 'LOC']:
                # Используем нормализованную форму
                normalized = span.normal if span.normal else span.text.lower()
                text = span.text

                # Создаем ключ для уникальности
                key = f"{span.type}:{normalized}"

                if key not in seen:
                    seen.add(key)
                    entities.append({
                        'text': text,
                        'type': span.type,
                        'normalized': normalized
                    })

        return entities

    except Exception as e:
        print(f"Ошибка NER: {e}")
        return []

In [ ]:
# Тестируем
test_entities = extract_entities_from_text(docs[3]['content'])
print("Результаты с нормализацией:")
for e in test_entities:
    print(f"  {e['text']} ({e['type']}) -> {e['normalized']}")

Результаты с нормализацией:
  Такафуми Кудо (PER) -> Такафуми Кудо
  Мияма (LOC) -> Мияма
  Yahoo News Japan (ORG) -> Yahoo News Japan
  Кудо (PER) -> Кудо
  Мичико Томиока (PER) -> Мичико Томиока
  Японии (LOC) -> Япония


## Разбиение на пропозиции

In [5]:
!pip install razdel
from razdel import sentenize

In [ ]:
def extract_sentences(text):
    """
    Разбиение на предложения через razdel
    """
    sents = [s.text for s in sentenize(text)]
    final_sents = []
    for sent in sents:
        new_sents = re.split(r'(?<=\.)\s+(?=[А-ЯЁA-Z])', sent)
        final_sents.extend(new_sents)
    return final_sents

In [ ]:
# Тестируем
test_sentences = extract_sentences(docs[0]['content'])
for e in test_sentences:
    print(e)

Врачи петербургской Елизаветинской больницы успешно прооперировали 100-летнюю женщину.
Из-за возраста пациентки проводить операцию было сложно, однако все прошло успешно, сообщает 24 июля пресс-служба медучреждения в соцсетях.
Опасный диагноз.
Долгожительница поступила в больницу с жалобами на боль и онемение в правой ноге.
Ее направили на обследование, врачи заподозрили тромбоз артерий конечности.
При подобных диагнозах действовать нужно быстро, буквально в течение 6 часов провести операцию и устранить тромб.
Иначе начнутся некротические процессы, которые в перспективе могут довести до гангрены и ампутации ноги.
Даже если некроза удалось бы избежать, в организме начались бы ишемические процессы и наступила интоксикация, а это уже представляет угрозу для жизни человека.
Наркоз мог убить.
Обследование подтвердило: у женщины тромбоз бедренных артерий со стенозом и атеросклеротическими бляшками.
Ее в экстренном порядке увезли в операционную.
Ситуацию осложнял возраст пациентки, а также со

# Создание графа

In [6]:
import networkx as nx
from sentence_transformers import SentenceTransformer

In [ ]:
# Модель для эмбеддингов
embedder = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
G = nx.MultiDiGraph()

statement_to_doc = {}  # stmt_id -> doc_id
doc_to_statements = defaultdict(list)  # doc_id -> [stmt_ids]
statement_entities = {}  # stmt_id -> [entities]
entity_to_statements = defaultdict(list)  # entity_key -> [stmt_ids]
statement_embeddings = {}  # stmt_id -> embedding

In [ ]:
def build_graph(documents):
    """
    Строит HLG из списка документов
    """
    for doc_idx, doc in enumerate(tqdm(documents, desc="Построение графа")):
        doc_id = f"doc_{doc_idx:04d}"
        doc['doc_id'] = doc_id

        # Получаем предложения
        sentences = extract_sentences(doc['content'])

        for sent_idx, sentence in enumerate(sentences):

            stmt_id = f"{doc_id}_stmt_{sent_idx:03d}"

            # Получаем эмбеддинг
            emb = embedder.encode(sentence)
            statement_embeddings[stmt_id] = emb

            # Извлекаем сущности
            entities = extract_entities_from_text(sentence)

            # Добавляем узел-утверждение в граф
            G.add_node(stmt_id,
                      type='statement',
                      text=sentence,
                      doc_id=doc_id,
                      doc_title=doc['title'],
                      embedding=emb)

            # Сохраняем связи
            statement_to_doc[stmt_id] = doc_id
            doc_to_statements[doc_id].append(stmt_id)
            statement_entities[stmt_id] = entities

            # Добавляем узлы-сущности и связи
            for entity in entities:
                entity_key = f"{entity['type']}:{entity['normalized']}"
                entity_id = f"entity_{hash(entity_key) % 1000000}"

                if not G.has_node(entity_id):
                    G.add_node(entity_id,
                              type='entity',
                              name=entity['text'],
                              normalized=entity['normalized'],
                              entity_type=entity['type'])

                G.add_edge(stmt_id, entity_id, relation='mentions')
                G.add_edge(entity_id, stmt_id, relation='mentioned_in')
                entity_to_statements[entity_key].append(stmt_id)

    return {
        'graph': G,
        'statement_to_doc': statement_to_doc,
        'doc_to_statements': doc_to_statements,
        'statement_entities': statement_entities,
        'entity_to_statements': entity_to_statements,
        'statement_embeddings': statement_embeddings
    }

# Создание индекса

In [7]:
!pip install faiss-cpu
import faiss

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 85.3 MB/s eta 0:00:00


In [ ]:
def build_faiss_index(statement_embeddings):
    """
    Строит FAISS индекс для векторного поиска
    """
    statements = list(statement_embeddings.keys())
    embeddings = np.array([statement_embeddings[s] for s in statements]).astype('float32')

    # Нормализуем
    faiss.normalize_L2(embeddings)

    # Создаем индекс
    index = faiss.IndexFlatIP(embeddings.shape[1])
    index.add(embeddings)

    return index, statements

In [ ]:
def save_all(data):
    """
    Сохраняет все структуры и скачивает их
    """
    saved_files = []

    with open('graph.pkl', 'wb') as f:
        pickle.dump(data['graph'], f)
    saved_files.append('graph.pkl')

    with open('statement_to_doc.pkl', 'wb') as f:
        pickle.dump(data['statement_to_doc'], f)
    saved_files.append('statement_to_doc.pkl')

    with open('doc_to_statements.pkl', 'wb') as f:
        pickle.dump(data['doc_to_statements'], f)
    saved_files.append('doc_to_statements.pkl')

    with open('statement_entities.pkl', 'wb') as f:
        pickle.dump(data['statement_entities'], f)
    saved_files.append('statement_entities.pkl')

    with open('entity_to_statements.pkl', 'wb') as f:
        pickle.dump(data['entity_to_statements'], f)
    saved_files.append('entity_to_statements.pkl')

    with open('statement_embeddings.pkl', 'wb') as f:
        pickle.dump(data['statement_embeddings'], f)
    saved_files.append('statement_embeddings.pkl')

    faiss.write_index(data['faiss_index'], 'faiss.index')
    saved_files.append('faiss.index')

    with open('faiss_statements.pkl', 'wb') as f:
        pickle.dump(data['faiss_statements'], f)
    saved_files.append('faiss_statements.pkl')

    print(f"Все данные сохранены")

    for file in saved_files:
        files.download(file)
        print(f"Скачан: {file}")

    print("Все файлы скачаны!")

# Запуск

In [ ]:
print("Построение графа...")
graph_data = build_graph(docs)

Построение графа...


Построение графа: 100%|██████████| 750/750 [13:25<00:00,  1.07s/it]


In [ ]:
print("Построение FAISS индекса...")
index, statements = build_faiss_index(graph_data['statement_embeddings'])
graph_data['faiss_index'] = index
graph_data['faiss_statements'] = statements

Построение FAISS индекса...


In [ ]:
print("Статистика:")
print(f"   Документов: {len(docs)}")
print(f"   Утверждений: {len(graph_data['statement_embeddings'])}")
print(f"   Сущностей: {graph_data['graph'].number_of_nodes() - len(graph_data['statement_embeddings'])}")
print(f"   Ребер: {graph_data['graph'].number_of_edges()}")

Статистика:
   Документов: 750
   Утверждений: 24027
   Сущностей: 2852
   Ребер: 11244


In [ ]:
print("Сохранение...")
save_all(graph_data)

Сохранение...
Все данные сохранены


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Скачан: graph.pkl


<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

Скачан: faiss.index


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Скачан: faiss_statements.pkl
Все файлы скачаны!


# StatementGraphRAG

In [8]:
from sentence_transformers import CrossEncoder

In [38]:
class StatementGraphRAG:
    def __init__(self, ):
        with open(f'graph.pkl', 'rb') as f:
            self.graph = pickle.load(f)

        with open(f'statement_embeddings.pkl', 'rb') as f:
            self.statement_embeddings = pickle.load(f)

        with open(f'entity_to_statements.pkl', 'rb') as f:
            self.entity_to_statements = pickle.load(f)

        self.faiss_index = faiss.read_index(f'faiss.index')

        with open(f'faiss_statements.pkl', 'rb') as f:
            self.faiss_statements = pickle.load(f)

        self.embedder = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')
        self.reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

    def keyword_search(self, query, top_k=10):
        """
        Поиск по сущностям
        """
        query_entities = extract_entities_from_text(query)
        results = set()

        for qe in query_entities:
            key = f"{qe['type']}:{qe['normalized']}"
            if key in self.entity_to_statements:
                results.update(self.entity_to_statements[key])

        return list(results)[:top_k]

    def vector_search(self, query, top_k=10):
        """
        Векторный поиск
        """
        q_emb = self.embedder.encode(query).astype('float32').reshape(1, -1)
        faiss.normalize_L2(q_emb)

        scores, indices = self.faiss_index.search(q_emb, top_k)
        return [self.faiss_statements[i] for i in indices[0] if i >= 0]

    def retrieve(self, query, top_k=5):
        """
        Полный поиск
        """
        # Keyword + Vector
        s_kw = self.keyword_search(query, 10)
        s_vss = self.vector_search(query, 10)
        candidates = list(set(s_kw + s_vss))

        # Reranking
        pairs = [[query, self.graph.nodes[s]['text']] for s in candidates]
        scores = self.reranker.predict(pairs)

        results = sorted(zip(candidates, scores), key=lambda x: x[1], reverse=True)
        return [r[0] for r in results[:top_k]]

In [39]:
srag = StatementGraphRAG()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [41]:
print("=" * 60)
print("СТАТИСТИКА ГРАФА")
print("=" * 60)

# Основная статистика
print(f"\n Общая информация:")
print(f"  - Всего узлов: {srag.graph.number_of_nodes()}")
print(f"  - Всего ребер: {srag.graph.number_of_edges()}")

# Типы узлов
node_types = {}
for node, data in srag.graph.nodes(data=True):
    ntype = data.get('type', 'unknown')
    node_types[ntype] = node_types.get(ntype, 0) + 1

print(f"\n Типы узлов:")
for ntype, count in node_types.items():
    print(f"   - {ntype}: {count}")

if 'entity' in node_types:
    entity_types = {}
    for node, data in srag.graph.nodes(data=True):
        if data.get('type') == 'entity':
            etype = data.get('entity_type', 'UNKNOWN')
            entity_types[etype] = entity_types.get(etype, 0) + 1

    print(f"\n Типы сущностей:")
    for etype, count in entity_types.items():
        print(f"   - {etype}: {count}")

# Статистика по документам
doc_nodes = [n for n, d in srag.graph.nodes(data=True) if d.get('type') == 'statement']
if doc_nodes:
    # Группируем по документам
    doc_stats = defaultdict(int)
    for node in doc_nodes:
        doc_id = srag.graph.nodes[node].get('doc_id', 'unknown')
        doc_stats[doc_id] += 1

    print(f"\n По документам:")
    print(f"  - Всего документов: {len(doc_stats)}")
    print(f"  - Среднее утверждений на документ: {len(doc_nodes)/len(doc_stats):.1f}")
    print(f"  - Максимум утверждений в документе: {max(doc_stats.values())}")
    print(f"  - Минимум утверждений в документе: {min(doc_stats.values())}")

СТАТИСТИКА ГРАФА

 Общая информация:
  - Всего узлов: 26879
  - Всего ребер: 11244

 Типы узлов:
   - statement: 24027
   - entity: 2852

 Типы сущностей:
   - ORG: 1045
   - PER: 1289
   - LOC: 518

 По документам:
  - Всего документов: 750
  - Среднее утверждений на документ: 32.0
  - Максимум утверждений в документе: 161
  - Минимум утверждений в документе: 3


# Запросы по StatementGraphRAG

In [ ]:
def search(query):
    results = srag.retrieve(query)
    for i, stmt_id in enumerate(results, 1):
        text = srag.graph.nodes[stmt_id]['text']
        doc_title = srag.graph.nodes[stmt_id]['doc_title']
        print(f"{i}. [{doc_title}]\n{text}\n")

In [ ]:
search('сколько часов надо спать в день?')

1. [Диетолог Обанина объяснила, почему по утрам нет аппетита, а вечером хочется есть]
Попробуйте сдвинуть свой график сна на пораньше — постарайтесь ложиться спать до 23 часов, уделяя сну 7-8 часов.

2. [Врач Юзуп рассказала, что делать, если переел в праздники]
Засыпать нужно не позднее 23 часов, а продолжительность сна должна быть не менее 9 часов.

3. [Стало известно, сколько дополнительных калорий в день мы съедаем из-за недосыпа]
На самом деле, спать можно ложиться хоть под утро, но главное — спать достаточное количество часов и соблюдать постоянный режим — если вы — «сова», то всегда ложиться спать примерно в одно и то же время, даже если будет далеко за полночь.

4. [Чем дыня полезна для организма, как правильно едят дыню, что полезнее — дыня или арбуз]
Еще одно важное правило: старайтесь лакомиться дыней не менее, чем за 2-3 часа до сна, — рассказывает Елена Саблина.

5. [На Солнце растет количество вспышек: астрономы рассказали, когда ждать мощных магнитных бурь]
Увеличьте вре

In [ ]:
search('как бороться с мигренью?')

1. [4 напитка, которые могут спровоцировать приступ головной боли]
Как итог — вы будете страдать от всех симптомов обезвоживания, в том числе и от сильной головной боли.

2. [Астрономы предупредили о высоком риске магнитных бурь в конце недели]
Людям, которые мучатся в периоды бурь из-за головной боли, имеет смысл заранее купить обезболивающее.

3. [5 пищевых привычек, которые могут вызвать мучительную головную боль]
Неправильное питание запросто может вызвать или усугубить головную боль.

4. [Московские ученые создали фантом, который поможет искать тромбы в сосудах и опухоли в груди]
ФУ) пришла к выводу, что определенная головная боль может быть предвестником инсульта.

5. [Петербурженку с желудочным кровотечением лечили в коридоре лекарствами за свой счет › Статьи и новости › Доктор.]
- С момента поступления у меня очень сильно болит голова.



In [ ]:
search('польза пива')

1. [8 питьевых привычек, которые заставляют наш мозг стариться быстрее]
И снова про алкоголь.

2. [4 напитка, которые могут спровоцировать приступ головной боли]
Пиво или вино.

3. [Как убрать пивной живот: 5 простых советов диетолога]
И что важно — пиво тут же уходит в тот самый «брюшной жир».

4. [10 продуктов, которые вредят зубам больше, чем сахар, — среди них есть и полезные]
Любые алкогольные напитки.

5. [Эндокринолог Яровова назвала 9 причин, почему никак не получается похудеть]
Алкоголь.



In [ ]:
search('Стоит ли делать прививки?')

1. [Американские врачи объяснили, что нужно есть при заражении омикроном]
Врачи говорят о том, что спасти от него не сможет даже прививка — хоть вакцинированные и легче переносят болезнь.

2. [6 способов повысить иммунитет без лекарств и БАДов]
Сегодня есть отличное средство защиты — прививки.

3. [К 2030 году вакцина от рака груди сможет защитить от болезни или остановить ее]
Если все получится, вакцина станет первой в своем роде.

4. [В СМИ сообщили об ухудшении состояния Жириновского]
Врачи уверены: такой подход к вакцинации может убить иммунитет.

5. [Минздрав прокомментировал сообщения СМИ о смерти Владимира Жириновского]
Врачи уверены: такой подход к вакцинации может убить иммунитет.



# TopicGraphRAG

In [9]:
from sklearn.metrics.pairwise import cosine_similarity

In [10]:
# !pip install transformers torch accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 7.2 MB/s eta 0:00:00


In [11]:
# from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
# import torch
# import re

In [12]:
import os
import json

In [29]:
class TopicGraphRAG:
    def __init__(self, rebuild_topics=False):
        """
        Инициализация
        """
        # Загружаем граф
        with open('graph.pkl', 'rb') as f:
            self.graph = pickle.load(f)

        # Загружаем эмбеддинги
        with open('statement_embeddings.pkl', 'rb') as f:
            self.statement_embeddings = pickle.load(f)

        # Загружаем связи сущностей
        with open('entity_to_statements.pkl', 'rb') as f:
            self.entity_to_statements = pickle.load(f)

        # Загружаем FAISS индекс
        self.faiss_index = faiss.read_index('faiss.index')
        with open('faiss_statements.pkl', 'rb') as f:
            self.faiss_statements = pickle.load(f)

        # Модели для эмбеддингов и ранжирования
        self.embedder = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')
        self.reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

        # Загружаем документы
        with open('documents.pkl', 'rb') as f:
            docs = pickle.load(f)

        # Получаем все doc_id из графа
        doc_ids_from_graph = set()
        for node, data in self.graph.nodes(data=True):
            if data.get('type') == 'statement':
                doc_id = data.get('doc_id')
                if doc_id:
                    doc_ids_from_graph.add(doc_id)

        # Сопоставляем документы с doc_id из графа
        self.docs = {}
        doc_list = list(docs)

        for i, doc_id in enumerate(sorted(doc_ids_from_graph)[:len(doc_list)]):
            doc = doc_list[i]
            doc['doc_id'] = doc_id
            self.docs[doc_id] = doc

        # Инициализируем Natasha для NER
        self.segmenter = Segmenter()
        self.emb = NewsEmbedding()
        self.morph_tagger = NewsMorphTagger(self.emb)
        self.ner_tagger = NewsNERTagger(self.emb)
        self.morph_vocab = MorphVocab()

        # Кэши
        self.neighbor_cache = {}

        # Загружаем или строим темы
        topics_file = 'topics_cache.pkl'

        if rebuild_topics or not os.path.exists(topics_file):
            self.build_topics()
            self._save_topics(topics_file)
        else:
            self._load_topics(topics_file)

        self.build_topic_index()

    def _save_topics(self, filename):
        """
        Сохраняет построенные темы в файл
        """
        topics_data = {
            'topics': self.topics,
            'topic_embeddings': self.topic_embeddings,
            'topic_to_statements': dict(self.topic_to_statements),
            'statement_to_topic': self.statement_to_topic
        }

        with open(filename, 'wb') as f:
            pickle.dump(topics_data, f)

    def _load_topics(self, filename):
        """
        Загружает темы из файла
        """
        with open(filename, 'rb') as f:
            topics_data = pickle.load(f)

        self.topics = topics_data['topics']
        self.topic_embeddings = topics_data['topic_embeddings']
        self.topic_to_statements = defaultdict(list, topics_data['topic_to_statements'])
        self.statement_to_topic = topics_data['statement_to_topic']

    def build_topics(self):
        """
        Строит темы, используя заголовки документов
        """
        self.topics = {}
        self.topic_embeddings = {}
        self.topic_to_statements = defaultdict(list)
        self.statement_to_topic = {}

        # Группируем утверждения по doc_id
        doc_statements = defaultdict(list)
        for node, data in self.graph.nodes(data=True):
            if data.get('type') == 'statement':
                doc_id = data.get('doc_id')
                if doc_id:
                    doc_statements[doc_id].append(node)

        # Создаем темы
        topic_count = 0
        for doc_id, statements in doc_statements.items():
            if len(statements) < 2:
                continue

            topic_id = f"topic_{topic_count:04d}"
            topic_count += 1

            # Берем заголовок из документа
            doc_info = self.docs.get(doc_id, {})
            title = doc_info.get('title', 'Без названия')

            # Считаем эмбеддинг темы
            valid_embs = [self.statement_embeddings[s] for s in statements if s in self.statement_embeddings]
            if valid_embs:
                topic_emb = np.mean(valid_embs, axis=0)
            else:
                continue

            self.topics[topic_id] = {
                'doc_id': doc_id,
                'title': title,
                'num_statements': len(statements)
            }
            self.topic_embeddings[topic_id] = topic_emb
            self.topic_to_statements[topic_id] = statements

            for stmt_id in statements:
                self.statement_to_topic[stmt_id] = topic_id

    def build_topic_index(self):
        """
        Строит FAISS индекс для быстрого поиска по темам
        """
        if not self.topics:
            return

        self.topic_ids = list(self.topic_embeddings.keys())
        topic_embs = np.array([self.topic_embeddings[tid] for tid in self.topic_ids]).astype('float32')

        faiss.normalize_L2(topic_embs)

        self.topic_index = faiss.IndexFlatIP(topic_embs.shape[1])
        self.topic_index.add(topic_embs)

    def extract_entities_from_query(self, query):
        """
        Извлекает сущности из запроса с помощью Natasha
        """
        doc = Doc(query)
        doc.segment(self.segmenter)
        doc.tag_morph(self.morph_tagger)
        doc.tag_ner(self.ner_tagger)

        for span in doc.spans:
            span.normalize(self.morph_vocab)

        entities = []
        for span in doc.spans:
            if span.type in ['ORG', 'PER', 'LOC']:
                normalized = span.normal if span.normal else span.text.lower()
                entities.append(f"{span.type}:{normalized}")

        return entities

    def top_down_search(self, query, top_k=10):
        """
        Поиск по темам (Top-Down)
        """
        if not hasattr(self, 'topic_index'):
            return []

        q_emb = self.embedder.encode(query).astype('float32').reshape(1, -1)
        faiss.normalize_L2(q_emb)

        scores, indices = self.topic_index.search(q_emb, top_k)

        return [self.topic_ids[i] for i in indices[0] if i >= 0]

    def bottom_up_search(self, query, top_k=20):
        """
        Поиск по сущностям (Bottom-Up)
        """
        query_entities = self.extract_entities_from_query(query)

        if not query_entities:
            return []

        statements = set()
        for entity_key in query_entities:
            if entity_key in self.entity_to_statements:
                statements.update(self.entity_to_statements[entity_key])

        return list(statements)[:top_k]

    def get_neighbors(self, stmt_id, max_neighbors=10):
        """
        Находит соседей утверждения через общие сущности
        """
        if stmt_id in self.neighbor_cache:
            return self.neighbor_cache[stmt_id]

        neighbors = set()
        entities = [n for n in self.graph.neighbors(stmt_id)
                   if self.graph.nodes[n].get('type') == 'entity']

        for entity in entities:
            for neighbor in self.graph.neighbors(entity):
                if (neighbor != stmt_id and
                    self.graph.nodes[neighbor].get('type') == 'statement'):
                    neighbors.add(neighbor)

        result = list(neighbors)[:max_neighbors]
        self.neighbor_cache[stmt_id] = result
        return result

    def attention_weighted_path(self, path_statements, query_emb):
        """
        Взвешенное по вниманию среднее
        """
        if not path_statements:
            return None

        path_embs = []
        for stmt_id in path_statements:
            if stmt_id in self.statement_embeddings:
                path_embs.append(self.statement_embeddings[stmt_id])

        if not path_embs:
            return None

        path_embs = np.array(path_embs)

        similarities = cosine_similarity([query_emb], path_embs)[0]
        weights = np.exp(similarities) / (np.sum(np.exp(similarities)) + 1e-10)

        path_emb = np.sum(weights.reshape(-1, 1) * path_embs, axis=0)

        return path_emb

    def beam_search(self, seed_statements, query, beam_width=3, max_depth=2):
        """
        Лучевой поиск по графу
        """
        query_emb = self.embedder.encode(query)

        paths = [[s] for s in seed_statements[:beam_width*2]]
        all_visited = set(seed_statements)

        for depth in range(max_depth):
            new_paths = []

            for path in paths:
                current = path[-1]
                neighbors = self.get_neighbors(current, max_neighbors=5)

                for neighbor in neighbors:
                    if neighbor not in path:
                        new_paths.append(path + [neighbor])
                        all_visited.add(neighbor)

            if not new_paths:
                break

            path_scores = []
            for path in new_paths:
                path_emb = self.attention_weighted_path(path, query_emb)
                if path_emb is not None:
                    score = cosine_similarity([query_emb], [path_emb])[0][0]
                    path_scores.append((path, score))

            path_scores.sort(key=lambda x: x[1], reverse=True)
            paths = [p for p, _ in path_scores[:beam_width]]

        return list(all_visited)

    def retrieve(self, query, top_k=5, beam_width=3, max_depth=2):
        """
        Полный пайплайн TopicGraphRAG
        """
        # Поиск по темам
        topic_results = self.top_down_search(query, top_k=10)

        topic_statements = set()
        for topic_id in topic_results:
            topic_statements.update(self.topic_to_statements[topic_id])

        # Поиск по сущностям
        entity_statements = self.bottom_up_search(query, top_k=20)

        # Объединение начальных кандидатов
        seed_statements = list(set(topic_statements) | set(entity_statements))

        # Лучевой поиск
        if seed_statements and max_depth > 0:
            beam_statements = self.beam_search(seed_statements, query, beam_width, max_depth)
            all_statements = list(set(seed_statements + beam_statements))
        else:
            all_statements = seed_statements

        # Ранжирование
        if not all_statements:
            return []

        if len(all_statements) > 100:
            all_statements = all_statements[:100]

        texts = []
        valid_stmts = []
        for stmt_id in all_statements:
            if stmt_id in self.graph.nodes:
                text = self.graph.nodes[stmt_id].get('text', '')
                if text:
                    texts.append(text)
                    valid_stmts.append(stmt_id)

        if not texts:
            return []

        pairs = [[query, text] for text in texts]
        scores = self.reranker.predict(pairs)

        results = sorted(zip(valid_stmts, scores), key=lambda x: x[1], reverse=True)

        return results[:top_k]

In [30]:
trag = TopicGraphRAG(rebuild_topics=True)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [31]:
print(f"Всего тем: {len(trag.topics)}")
print(f"Всего утверждений в графе: {len(trag.statement_embeddings)}")

Всего тем: 750
Всего утверждений в графе: 24027


In [34]:
print("\nПримеры тем:")
for i, (topic_id, info) in enumerate(list(trag.topics.items())[:3]):
    print(f"\n{i+1}. {info['title']}")
    print(f"   Документ: {info['doc_id']}")
    print(f"   Утверждений: {info['num_statements']}")

    stmts = trag.topic_to_statements[topic_id][:2]
    for j, stmt_id in enumerate(stmts, 1):
        text = trag.graph.nodes[stmt_id].get('text', '')
        print(f"   Утв.{j}: {text}")


Примеры тем:

1. «Могла не выдержать наркоз»: в Елизаветинской больнице спасли 100-летнюю пациентку с тромбами в ноге
   Документ: doc_0000
   Утверждений: 22
   Утв.1: Врачи петербургской Елизаветинской больницы успешно прооперировали 100-летнюю женщину.
   Утв.2: Из-за возраста пациентки проводить операцию было сложно, однако все прошло успешно, сообщает 24 июля пресс-служба медучреждения в соцсетях.

2. 5 привычек, которые день за днем незаметно повышают уровень холестерина
   Документ: doc_0001
   Утверждений: 38
   Утв.1: Вы можете даже не подозревать, что ваши сосуды облеплены воскообразным веществом.
   Утв.2: Речь, конечно, о холестерине, который не зря относят к одному из тихих убийц вашего здоровья.

3. Для бодрости и стройности: эндокринолог поделилась 8 рецептами завтраков, которые готовит сама
   Документ: doc_0002
   Утверждений: 35
   Утв.1: Есть люди, которые предпочитают вообще не завтракать, максимум по пути на работу перехватят стакан кофе и булочку из ближайшей пек

# Запросы по TopicGraphRAG

In [35]:
def search_trag(query):
    """
    Поиск через TopicGraphRAG
    """
    results = trag.retrieve(query)
    for i, (stmt_id, score) in enumerate(results, 1):
        text = trag.graph.nodes[stmt_id]['text']
        doc_title = trag.graph.nodes[stmt_id]['doc_title']
        print(f"{i}. [Релевантность: {score:.3f}] [{doc_title}]\n{text}\n")

In [36]:
search_trag("сколько часов надо спать в день?")

1. [Релевантность: 8.888] [3 лайфхака, которые помогут проснуться рано утром в хорошем настроении]
Сон на левом боку, в свою очередь, уменьшает количество эпизодов рефлюкса.

2. [Релевантность: 8.825] [Не ешьте творог на ночь: эндокринолог предупредила, почему это опасно]
Только обязательно надо учитывать то, что вы ели в течение дня, предупреждает врач.

3. [Релевантность: 8.792] [Диетолог Калиник рассказала, что пить на ночь, чтобы быстрее заснуть]
Если вы стали замечать, что вам трудно уснуть, то не спешите прибегать к снотворным, говорит британский диетолог Ева Калиник.

4. [Релевантность: 8.764] [Сначала пьем, потом встаем: как правильно начинать утро — 4 последовательных шага]
Мало кто может похвастаться здоровым сном на постоянной основе, да и необходимость в спешке собираться на работу и ехать по пробкам настроения тоже не добавляет.

5. [Релевантность: 8.752] [3 лайфхака, которые помогут проснуться рано утром в хорошем настроении]
Особенно если вы поздно легли и вставать совсе

# Сохранение графа в формате GEXF

In [44]:
def save_for_gephi(G, filename='graph.gexf'):
    """
    Сохраняет граф в формате GEXF
    """

    G_clean = nx.MultiDiGraph()

    for node, data in G.nodes(data=True):
        clean_data = {}
        for key, value in data.items():
            # Пропускаем эмбеддинги и другие numpy массивы
            if key != 'embedding' and not isinstance(value, np.ndarray):
                # Преобразуем в простые типы
                if isinstance(value, (str, int, float, bool)):
                    clean_data[key] = value
                elif isinstance(value, list):
                    # Список преобразуем в строку, если он не слишком длинный
                    if len(str(value)) < 100:
                        clean_data[key] = str(value)

        G_clean.add_node(node, **clean_data)

    # Копируем ребра
    for u, v, data in G.edges(data=True):
        clean_data = {}
        for key, value in data.items():
            if isinstance(value, (str, int, float, bool)):
                clean_data[key] = value

        G_clean.add_edge(u, v, **clean_data)

    # Сохраняем
    nx.write_gexf(G_clean, filename)
    print(f" Граф сохранен в {filename}")
    print(f" Узлов: {G_clean.number_of_nodes()}, Ребер: {G_clean.number_of_edges()}")

    return G_clean

# Используйте
save_for_gephi(srag.graph, '/content/graph.gexf')

 Граф сохранен в /content/graph.gexf
 Узлов: 26879, Ребер: 11244
